# Manu AI Chatbot - Document Ingestion Demo

This notebook demonstrates how to load a PDF document, chunk it using LangChain text splitters, generate vector embeddings, and store them into Supabase Vector Store.

In [ ]:
import { SupabaseVectorStore } from "@langchain/community/vectorstores/supabase";
import { OpenAIEmbeddings } from "@langchain/openai";
import { createClient } from "@supabase/supabase-js";
import dotenv from "dotenv";

dotenv.config();

const embeddings = new OpenAIEmbeddings({
  model: "text-embedding-3-small",
});

const supabaseClient = createClient(
  process.env.SUPABASE_URL ?? '',
  process.env.SUPABASE_SERVICE_ROLE_KEY ?? '',
);

const vectorStore = new SupabaseVectorStore(embeddings, {
  client: supabaseClient,
  tableName: 'documents',
  queryName: 'match_documents',
});

In [ ]:
import { PDFLoader } from "@langchain/community/document_loaders/fs/pdf";
import { RecursiveCharacterTextSplitter } from "@langchain/textsplitters";
import { reduceDocs } from "./src/shared/state.js";

// Path to sample PDF document
const pdfPath = "./test_docs/sample.pdf";

const loader = new PDFLoader(pdfPath);
const docs = await loader.load();

const textSplitter = new RecursiveCharacterTextSplitter({
  chunkSize: 1000,
  chunkOverlap: 200,
});

const docSplits = await textSplitter.splitDocuments(docs);
const finalDocs = reduceDocs([], docSplits);

console.log(`Prepared ${finalDocs.length} chunks for vector indexing.`);

In [ ]:
// Index documents into Supabase
await vectorStore.addDocuments(finalDocs);
console.log("Documents successfully indexed in Supabase Vector Store!");

In [ ]:
// Query vector store for similarity search
const results = await vectorStore.similaritySearch("What are the core capabilities?", 3);
console.log("Search Results:", results);